In [51]:
import pandas as pd
import numpy as np
import re

In [52]:
train=pd.read_parquet('../data/train_sample.parquet')
val=pd.read_parquet('../data/val_sample.parquet')
test=pd.read_parquet('../data/test_full.parquet')

In [53]:
train

,PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,TOTAL_SENTENCE
0,1578145,2864,590.551180,DGMJDFKDRFU Swallow Tailed Coat Men Circus Cos...
1,1060207,8500,100.000000,"Offex Cat 5e Blue Ethernet Patch Cable, Snagle..."
2,1768955,1436,590.551180,UPKOCH Cookie Press Kit Multifunctional DIY Bi...
3,2691742,2917,1259.842518,SM TRENDZ Woven Paithani Beautiful Trending Ja...
4,1007611,1,750.000000,Bon-mots of Charles Lamb and Douglas Jerrold
...,...,...,...,...
59995,1049008,3224,1400.000000,SALOMON Men's XA Pro 3D CS WP Trail Running Sh...
59996,2267473,10389,590.551180,VAYDO KN95/N95 5 Ply Face Mask Breathable New ...
59997,574819,21,598.424000,Pictorial Photo
59998,2424830,2211,669.291338,COVER CAPITAL Back Camera Lens Tempered Glass ...


In [54]:
print(f"Train : {train.shape}")
print(f"Val   : {val.shape}")
print(f"Test  : {test.shape}")

Train : (60000, 4)
Val   : (5000, 4)
Test  : (108661, 4)


# HTML cleaning

In [89]:
HTML_TAG     = re.compile(r"<[^>]+>")
HTML_ENTITY  = re.compile(r"&[a-z]+;|&#\d+;", re.IGNORECASE)
MULTI_SPACE  = re.compile(r"\s+")

In [90]:
EMOJI_PAT    = re.compile("["
                u"\U0001F600-\U0001F64F"  # emoticons
                u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                u"\U0001F680-\U0001F6FF"  # transport & map
                u"\U0001F1E0-\U0001F1FF"  # flags
                u"\U00002500-\U00002BEF"  # chinese/japanese/korean characters
                u"\U00002702-\U000027B0"  # dingbats
                u"\U000024C2-\U0001F251"
                "]+", flags=re.UNICODE)

SPECIAL_PAT  = re.compile(r"[^a-zA-Z0-9\s\.\,\-]")  # keep letters, numbers, space, period, comma, hyphen

In [91]:


def clean_text(text: str) -> str:
    text = HTML_TAG.sub(" ", text)
    text = HTML_ENTITY.sub(" ", text)
    text = text.replace("\\n", " ").replace("\n", " ").replace("\r", " ")  # newlines
    text = EMOJI_PAT.sub(" ", text)                                          # emojis
    text = SPECIAL_PAT.sub(" ", text)                                        # special chars
    text = text.lower().strip()
    text = MULTI_SPACE.sub(" ", text)
    return text

# regex based features

In [92]:
CM_PAT     = re.compile(r"(\d+\.?\d*)\s*cm", re.IGNORECASE)
INCH_PAT   = re.compile(r"(\d+\.?\d*)\s*(inch|inches|\")", re.IGNORECASE)
MM_PAT     = re.compile(r"(\d+\.?\d*)\s*mm", re.IGNORECASE)
METER_PAT  = re.compile(r"(\d+\.?\d*)\s*(meter|metres?|m)\b", re.IGNORECASE)
FEET_PAT   = re.compile(r"(\d+\.?\d*)\s*(feet|foot|ft)", re.IGNORECASE)
ANY_NUM    = re.compile(r"\d+\.?\d*")
# Dimension keywords
DIM_KEYWORDS = re.compile(
    r"\b(length|width|height|depth|size|dimension|long|diameter|radius)\b",
    re.IGNORECASE
)

In [93]:
def has_unit(text: str) -> int:
    return int(bool(CM_PAT.search(text) or INCH_PAT.search(text) or
                    MM_PAT.search(text) or METER_PAT.search(text) or
                    FEET_PAT.search(text)))


In [94]:
def count_units(text: str) -> int:
    """How many total unit mentions are in the description"""
    return (len(CM_PAT.findall(text)) + len(INCH_PAT.findall(text)) +
            len(MM_PAT.findall(text)) + len(METER_PAT.findall(text)) +
            len(FEET_PAT.findall(text)))

In [95]:
def first_numeric(text: str) -> float:
    """First number mentioned in the description"""
    match = ANY_NUM.search(text)
    return float(match.group()) if match else -1.0

In [96]:
def max_numeric(text: str) -> float:
    """Largest number in description — often the main dimension"""
    nums = ANY_NUM.findall(text)
    return max([float(n) for n in nums]) if nums else -1.0

In [97]:
def num_count(text: str) -> int:
    """How many numbers appear in description"""
    return len(ANY_NUM.findall(text))


In [98]:
def first_cm_value(text: str) -> float:
    """First cm value found — most likely to be a length"""
    match = CM_PAT.search(text)
    return float(match.group(1)) if match else -1.0

In [99]:
def first_inch_value(text: str) -> float:
    match = INCH_PAT.search(text)
    return float(match.group(1)) if match else -1.0

In [100]:
def has_dim_keyword(text: str) -> int:
    return int(bool(DIM_KEYWORDS.search(text)))

In [101]:
def count_dim_keywords(text: str) -> int:
    return len(DIM_KEYWORDS.findall(text))

In [102]:
def word_count(text: str) -> int:
    return len(text.split())

In [103]:
def char_count(text: str) -> int:
    return len(text)

In [104]:
def unit_type_flag(text: str) -> str:
    """
    Which unit system dominates in this description?
    Returns: 'metric', 'imperial', 'both', 'none'
    """
    has_metric   = bool(CM_PAT.search(text) or MM_PAT.search(text) or METER_PAT.search(text))
    has_imperial = bool(INCH_PAT.search(text) or FEET_PAT.search(text))
    if has_metric and has_imperial:
        return "both"
    elif has_metric:
        return "metric"
    elif has_imperial:
        return "imperial"
    return "none"

In [105]:
def apply_features(df, has_target=True):
    
    # Step 1: Clean text
    print("  Cleaning HTML...")
    df["CLEAN_TEXT"] = df["TOTAL_SENTENCE"].apply(clean_text)

    # Step 2: Auxiliary features
    print("  Extracting regex features...")
    df["HAS_UNIT"]          = df["CLEAN_TEXT"].apply(has_unit)
    df["UNIT_COUNT"]        = df["CLEAN_TEXT"].apply(count_units)
    df["FIRST_NUM"]         = df["CLEAN_TEXT"].apply(first_numeric)
    df["MAX_NUM"]           = df["CLEAN_TEXT"].apply(max_numeric)
    df["NUM_COUNT"]         = df["CLEAN_TEXT"].apply(num_count)
    df["FIRST_CM"]          = df["CLEAN_TEXT"].apply(first_cm_value)
    df["FIRST_INCH"]        = df["CLEAN_TEXT"].apply(first_inch_value)
    df["HAS_DIM_KEYWORD"]   = df["CLEAN_TEXT"].apply(has_dim_keyword)
    df["DIM_KEYWORD_COUNT"] = df["CLEAN_TEXT"].apply(count_dim_keywords)
    df["WORD_COUNT"]        = df["CLEAN_TEXT"].apply(word_count)
    df["CHAR_COUNT"]        = df["CLEAN_TEXT"].apply(char_count)
    df["UNIT_TYPE"]         = df["CLEAN_TEXT"].apply(unit_type_flag)

    # Step 3: Encode UNIT_TYPE
    unit_map = {"none": 0, "metric": 1, "imperial": 2, "both": 3}
    df["UNIT_TYPE_ENC"] = df["UNIT_TYPE"].map(unit_map)

    # Step 4: log1p target
    if has_target:
        print("  Applying log1p to target...")
        df["PRODUCT_LENGTH_LOG"] = np.log1p(df["PRODUCT_LENGTH"])

    return df

In [106]:
print("\nProcessing train...")
train_fe = apply_features(train, has_target=True)


Processing train...
  Cleaning HTML...
  Extracting regex features...
  Applying log1p to target...


In [107]:
train_fe

,PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,TOTAL_SENTENCE,CLEAN_TEXT,HAS_UNIT,UNIT_COUNT,FIRST_NUM,MAX_NUM,NUM_COUNT,FIRST_CM,FIRST_INCH,HAS_DIM_KEYWORD,DIM_KEYWORD_COUNT,WORD_COUNT,CHAR_COUNT,UNIT_TYPE,UNIT_TYPE_ENC,PRODUCT_LENGTH_LOG
0,1578145,2864,590.551180,DGMJDFKDRFU Swallow Tailed Coat Men Circus Cos...,dgmjdfkdrfu swallow tailed coat men circus cos...,0,0,36.0,36.0,6,-1.0,-1.0,1,2,181,1203,none,0,6.382748
1,1060207,8500,100.000000,"Offex Cat 5e Blue Ethernet Patch Cable, Snagle...","offex cat 5e blue ethernet patch cable, snagle...",1,1,5.0,5.0,2,-1.0,-1.0,0,0,12,68,imperial,2,4.615121
2,1768955,1436,590.551180,UPKOCH Cookie Press Kit Multifunctional DIY Bi...,upkoch cookie press kit multifunctional diy bi...,1,1,20.0,21.0,10,21.0,-1.0,1,1,95,620,metric,1,6.382748
3,2691742,2917,1259.842518,SM TRENDZ Woven Paithani Beautiful Trending Ja...,sm trendz woven paithani beautiful trending ja...,1,2,5.5,5.5,2,-1.0,-1.0,1,1,32,249,metric,1,7.139535
4,1007611,1,750.000000,Bon-mots of Charles Lamb and Douglas Jerrold,bon-mots of charles lamb and douglas jerrold,0,0,-1.0,-1.0,0,-1.0,-1.0,0,0,7,44,none,0,6.621406
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,1049008,3224,1400.000000,SALOMON Men's XA Pro 3D CS WP Trail Running Sh...,salomon men s xa pro 3d cs wp trail running sh...,1,2,3.0,1947.0,14,-1.0,-1.0,1,8,266,1544,metric,1,7.244942
59996,2267473,10389,590.551180,VAYDO KN95/N95 5 Ply Face Mask Breathable New ...,vaydo kn95 n95 5 ply face mask breathable new ...,0,0,95.0,99.9,9,-1.0,-1.0,0,0,105,682,none,0,6.382748
59997,574819,21,598.424000,Pictorial Photo,pictorial photo,0,0,-1.0,-1.0,0,-1.0,-1.0,0,0,2,15,none,0,6.395969
59998,2424830,2211,669.291338,COVER CAPITAL Back Camera Lens Tempered Glass ...,cover capital back camera lens tempered glass ...,0,0,20.0,20.0,1,-1.0,-1.0,0,0,90,582,none,0,6.507712


# EDA checkpoint

In [78]:
cm_pct   = (train_fe["FIRST_CM"] > 0).sum() / len(train_fe) * 100
inch_pct = (train_fe["FIRST_INCH"] > 0).sum() / len(train_fe) * 100
print(f"Rows with a CM value   : {cm_pct:.2f}%")
print(f"Rows with INCH value   : {inch_pct:.2f}%")

has_cm = train_fe[train_fe["FIRST_CM"] > 0]
corr_cm = has_cm["FIRST_CM"].corr(has_cm["PRODUCT_LENGTH"])
print(f"Corr(FIRST_CM, PRODUCT_LENGTH) : {corr_cm:.4f}")

has_inch = train_fe[train_fe["FIRST_INCH"] > 0]
corr_inch = has_inch["FIRST_INCH"].corr(has_inch["PRODUCT_LENGTH"])
print(f"Corr(FIRST_INCH, PRODUCT_LENGTH) : {corr_inch:.4f}")

print("\nUNIT_TYPE distribution:")
print(train_fe["UNIT_TYPE"].value_counts())

Rows with a CM value   : 7.77%
Rows with INCH value   : 18.14%
Corr(FIRST_CM, PRODUCT_LENGTH) : 0.0237
Corr(FIRST_INCH, PRODUCT_LENGTH) : -0.0009

UNIT_TYPE distribution:
UNIT_TYPE
none        43608
imperial     7895
metric       4972
both         3525
Name: count, dtype: int64


## Conclusion: These simply suggest that we need semantic understanding of the description as the values mentioned in the description have no connection with our target

In [108]:
print("Processing val...")
val_fe   = apply_features(val, has_target=True)

print("Processing test...")
test_fe  = apply_features(test, has_target=False)   # test has no PRODUCT_LENGTH

Processing val...
  Cleaning HTML...
  Extracting regex features...
  Applying log1p to target...
Processing test...
  Cleaning HTML...
  Extracting regex features...


In [109]:
cols = ["PRODUCT_ID", "PRODUCT_TYPE_ID", "PRODUCT_LENGTH", "CLEAN_TEXT", "PRODUCT_LENGTH_LOG"]

train_fe = train_fe[cols]
val_fe = val_fe[cols]
test_fe = test_fe[["PRODUCT_ID", "PRODUCT_TYPE_ID", "PRODUCT_LENGTH", "CLEAN_TEXT"]]

print(f"train_fe: {train_fe.shape}")
print(f"val_fe:   {val_fe.shape}")
print(f"test_fe:  {test_fe.shape}")

train_fe: (60000, 5)
val_fe:   (5000, 5)
test_fe:  (108661, 4)


In [110]:
train_fe

,PRODUCT_ID,PRODUCT_TYPE_ID,PRODUCT_LENGTH,CLEAN_TEXT,PRODUCT_LENGTH_LOG
0,1578145,2864,590.551180,dgmjdfkdrfu swallow tailed coat men circus cos...,6.382748
1,1060207,8500,100.000000,"offex cat 5e blue ethernet patch cable, snagle...",4.615121
2,1768955,1436,590.551180,upkoch cookie press kit multifunctional diy bi...,6.382748
3,2691742,2917,1259.842518,sm trendz woven paithani beautiful trending ja...,7.139535
4,1007611,1,750.000000,bon-mots of charles lamb and douglas jerrold,6.621406
...,...,...,...,...,...
59995,1049008,3224,1400.000000,salomon men s xa pro 3d cs wp trail running sh...,7.244942
59996,2267473,10389,590.551180,vaydo kn95 n95 5 ply face mask breathable new ...,6.382748
59997,574819,21,598.424000,pictorial photo,6.395969
59998,2424830,2211,669.291338,cover capital back camera lens tempered glass ...,6.507712


In [111]:
train_fe["CLEAN_TEXT"][0]

'dgmjdfkdrfu swallow tailed coat men circus costume burgundy sequins blazer for circus the stylish tuxedo coat with tails is made of lightweight sequins material,the sequins vest and pocket flaps are sewn directly to the jacket for decoration,the slim fit style makes you look modern and sharp on this shiny jacket for party costume, based on feedbacks of the ringmaster jacket,we have made adjustment in our size chart.you can order the bling costume jacket glitter blazer according your body chest for example,if your chest is 36 ,you can order the size s , about shipping imported,to usa,standard takes 6-8 days normally,expedited takes 3-4 days,if it is in stock,we will send it out asap, suitable for circus costume men,masquerade party,prom dancing,school musical,sequined coat for cosplay party,the joker costume for men,mardi gras season,goth nights and stage performance,ringmaster costume jacket.also suitable for magician and chorus conductor for event and show etc., the package include 1

In [112]:
train_fe.to_parquet('../data/train_fe.parquet', index=False)
val_fe.to_parquet('../data/val_fe.parquet', index=False)
test_fe.to_parquet('../data/test_fe.parquet', index=False)

print("Saved:")
print(f"  train_fe -> ../data/train_fe.parquet  {train_fe.shape}")
print(f"  val_fe   -> ../data/val_fe.parquet    {val_fe.shape}")
print(f"  test_fe  -> ../data/test_fe.parquet   {test_fe.shape}")

Saved:
  train_fe -> ../data/train_fe.parquet  (60000, 5)
  val_fe   -> ../data/val_fe.parquet    (5000, 5)
  test_fe  -> ../data/test_fe.parquet   (108661, 4)
